# Q-Learning no FrozenLake — Visualizacao Passo a Passo

Este notebook acompanha a aula de Aprendizado por Reforco.

Cada celula tem comentarios explicando o que o robo esta fazendo e por que.

Ao final voce vera:
- o mapa do ambiente com legendas completas
- a Q-table antes e depois do treinamento como mapa de calor
- como o epsilon vai diminuindo ao longo do treinamento
- o caminho que o robo aprende a percorrer em varios momentos
- a curva de aprendizado ao longo dos episodios com as tres fases
- o mapa final com a politica otima em setas
- a simulacao do robo treinado passo a passo
- comparacao entre diferentes configuracoes de hiperparametros

---

**Legenda das cores usadas nos mapas:**

| Cor | Letra | Significado |
|-----|-------|-------------|
| Verde | S | Inicio: posicao inicial do robo |
| Branco | F | Gelo seguro: o robo pode caminhar aqui |
| Vermelho | H | Buraco: cair aqui encerra o episodio sem recompensa |
| Azul | G | Objetivo: chegar aqui da recompensa +1 |
| Laranja | - | Celula visitada pelo robo naquele episodio |
| Amarelo | - | Posicao atual do robo durante a simulacao |

## Celula 1 — Instalacao e Importacoes

Instalamos e importamos as bibliotecas necessarias.

- `gymnasium`: fornece o ambiente FrozenLake pronto para uso
- `numpy`: cria e manipula a Q-table como uma matriz numerica
- `matplotlib`: gera todos os graficos, mapas e visualizacoes
- `random`: sorteia numeros para a estrategia de exploracao epsilon-greedy

In [ ]:
# Instala o gymnasium caso nao esteja disponivel no ambiente do Colab
!pip install gymnasium --quiet

# gymnasium: pacote com ambientes de RL prontos, incluindo o FrozenLake
import gymnasium as gym

# numpy: biblioteca para operacoes com arrays e matrizes
# a Q-table e uma matriz numpy de shape (16, 4)
import numpy as np

# matplotlib: biblioteca de visualizacao
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# random: usamos random.random() para implementar a estrategia epsilon-greedy
import random

# Faz com que os graficos aparecam diretamente no notebook
%matplotlib inline

print("Bibliotecas carregadas com sucesso.")

## Celula 2 — Criacao do Ambiente FrozenLake

O FrozenLake e um tabuleiro 4x4.

O robo comeca no estado 0 (canto superior esquerdo) e precisa chegar ao estado 15 (canto inferior direito).

Existem buracos nos estados 5, 7, 11 e 12. Cair em um deles encerra o episodio com recompensa zero.

Usamos `is_slippery=False` para que o robo va exatamente para onde ele decide ir.
No modo padrao (`is_slippery=True`) o gelo escorrega e o robo pode ir em outra direcao por acidente.

In [ ]:
# Cria o ambiente FrozenLake versao 4x4
# is_slippery=False: o robo se move exatamente na direcao escolhida (determinístico)
env = gym.make("FrozenLake-v1", is_slippery=False)

# Descobre quantos estados existem
# No FrozenLake 4x4 retorna 16 (celulas numeradas de 0 a 15)
n_estados = env.observation_space.n

# Descobre quantas acoes o agente pode tomar em cada estado
# No FrozenLake retorna 4: 0=esquerda, 1=baixo, 2=direita, 3=cima
n_acoes = env.action_space.n

# Nomes das acoes para exibir nos resultados
nomes_acoes = ["Esquerda", "Baixo", "Direita", "Cima"]

# Simbolos das acoes para desenhar no mapa da politica otima
simbolos_acoes = ["<", "v", ">", "^"]

# Estrutura do mapa: cada string e uma linha do tabuleiro
# S=Start, F=Frozen, H=Hole, G=Goal
mapa = [
    "SFFF",   # linha 0: estados  0,  1,  2,  3
    "FHFH",   # linha 1: estados  4,  5,  6,  7  (buracos em 5 e 7)
    "FFFH",   # linha 2: estados  8,  9, 10, 11  (buraco em 11)
    "HFFG",   # linha 3: estados 12, 13, 14, 15  (buraco em 12, objetivo em 15)
]

print(f"Numero de estados: {n_estados}")
print(f"Numero de acoes:   {n_acoes}")
print()
print("Mapeamento das acoes:")
for i, nome in enumerate(nomes_acoes):
    print(f"  {i} = {nome}  {simbolos_acoes[i]}")

## Celula 3 — Visualizacao do Mapa com Legendas

Antes de comecar o treinamento, vemos o tabuleiro completo.

O numero pequeno no canto de cada celula e o numero do estado (0 a 15).

O robo começa em S e precisa chegar a G sem cair nos buracos H.

In [ ]:
def desenhar_mapa(mapa, titulo="Mapa do FrozenLake", caminho=None, posicao_atual=None):
    """
    Desenha o tabuleiro FrozenLake com cores, numeros de estado e legendas.

    Parametros:
        mapa          : lista de strings com o layout do tabuleiro
        titulo        : texto exibido no topo da figura
        caminho       : lista de estados visitados naquele episodio (opcional)
        posicao_atual : estado onde o robo esta agora (opcional)
    """

    fig, ax = plt.subplots(figsize=(7, 7.5))

    # Cores de fundo para cada tipo de celula
    cores_celula = {
        "S": "#2ecc71",   # verde: inicio
        "F": "#ecf0f1",   # branco acinzentado: gelo seguro
        "H": "#e74c3c",   # vermelho: buraco
        "G": "#3498db",   # azul: objetivo
    }

    for linha_idx, linha in enumerate(mapa):
        for col_idx, celula in enumerate(linha):

            # Calcula o numero do estado a partir de linha e coluna
            estado = linha_idx * 4 + col_idx

            # Cor padrao da celula
            cor = cores_celula[celula]

            # Se esta celula faz parte do caminho percorrido, usa laranja
            if caminho is not None and estado in caminho:
                cor = "#f39c12"

            # Se e a posicao atual do robo, usa amarelo
            if posicao_atual is not None and estado == posicao_atual:
                cor = "#f1c40f"

            # Desenha o retangulo colorido da celula
            # O eixo y e invertido: estado 0 fica no canto superior esquerdo
            ret = plt.Rectangle(
                (col_idx, 3 - linha_idx),
                1, 1,
                color=cor, ec="#7f8c8d", lw=2
            )
            ax.add_patch(ret)

            # Numero do estado no canto superior esquerdo da celula
            ax.text(
                col_idx + 0.07, 3 - linha_idx + 0.87,
                str(estado),
                fontsize=9, color="#555555", va="top"
            )

            # Letra identificadora no centro da celula
            ax.text(
                col_idx + 0.5, 3 - linha_idx + 0.45,
                celula,
                fontsize=26, ha="center", va="center",
                fontweight="bold", color="#2c3e50"
            )

    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titulo, fontsize=13, fontweight="bold", pad=12)

    # Monta a legenda de acordo com o que esta sendo exibido
    legenda = [
        mpatches.Patch(color="#2ecc71", label="S — Inicio: posicao inicial do robo"),
        mpatches.Patch(color="#ecf0f1", label="F — Gelo seguro: o robo pode passar"),
        mpatches.Patch(color="#e74c3c", label="H — Buraco: episodio termina sem recompensa"),
        mpatches.Patch(color="#3498db", label="G — Objetivo: chegou, recompensa = +1"),
    ]
    if caminho is not None:
        legenda.append(mpatches.Patch(color="#f39c12", label="Caminho percorrido neste episodio"))
    if posicao_atual is not None:
        legenda.append(mpatches.Patch(color="#f1c40f", label="Posicao atual do robo"))

    ax.legend(
        handles=legenda, loc="upper center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=1, fontsize=10, frameon=True, edgecolor="#bdc3c7"
    )

    plt.tight_layout()
    plt.show()


# Exibe o mapa inicial, sem nenhuma marcacao de caminho
desenhar_mapa(mapa, titulo="Mapa do FrozenLake — antes do treinamento")

## Celula 4 — Definicao dos Hiperparametros com Decaimento de Epsilon

Hiperparametros sao as configuracoes que o professor define antes do treinamento.

Neste notebook usamos uma tecnica importante chamada **decaimento de epsilon**:

- No inicio do treinamento, epsilon e alto (proximo de 1.0): o robo explora muito, tentando acoes aleatorias para descobrir o ambiente.
- Com o passar dos episodios, epsilon diminui gradualmente.
- No final do treinamento, epsilon e baixo (proximo de 0.01): o robo aproveita o que aprendeu.

Isso resolve o dilema entre explorar (descobrir coisas novas) e explotar (usar o que ja sabe).

| Hiperparametro | Valor | O que faz |
|---|---|---|
| alpha | 0.8 | Quanto corrige a Q-table em cada passo |
| gamma | 0.99 | Quanto valoriza recompensas futuras |
| epsilon_inicio | 1.0 | Começa explorando quase sempre |
| epsilon_fim | 0.01 | No final quase sempre usa a tabela |
| decaimento | 0.001 | Velocidade de reducao do epsilon |
| episodios | 5000 | Total de tentativas de treinamento |

In [ ]:
# -------------------------------------------------------
# HIPERPARAMETROS — altere estes valores para experimentar
# -------------------------------------------------------

# Taxa de aprendizado: controla o quanto a Q-table e corrigida a cada atualizacao
# 0.8 = correcoes grandes e rapidas (bom para ambientes determinísticos)
alpha = 0.8

# Fator de desconto: quanto o robo valoriza recompensas futuras
# 0.99 = recompensas distantes no tempo ainda importam muito
# Necessario alto aqui pois a recompensa so chega ao final do caminho
gamma = 0.99

# Epsilon inicial: probabilidade de exploracao no começo do treinamento
# 1.0 = o robo escolhe acoes totalmente aleatorias no inicio
epsilon_inicio = 1.0

# Epsilon final: probabilidade de exploracao no final do treinamento
# 0.01 = apenas 1% de acoes aleatorias quando o treinamento converge
epsilon_fim = 0.01

# Velocidade de decaimento do epsilon a cada episodio
# A formula e: epsilon = max(epsilon_fim, epsilon * (1 - decaimento))
# 0.001 = reducao de 0.1% por episodio
taxa_decaimento = 0.001

# Numero total de episodios de treinamento
episodios = 5000

# Intervalo de episodios para salvar e exibir o caminho percorrido
intervalo_visual = 1000

print("Hiperparametros definidos:")
print(f"  alpha (taxa de aprendizado)  = {alpha}")
print(f"  gamma (fator de desconto)    = {gamma}")
print(f"  epsilon inicio               = {epsilon_inicio}")
print(f"  epsilon fim                  = {epsilon_fim}")
print(f"  taxa de decaimento           = {taxa_decaimento}")
print(f"  episodios totais             = {episodios}")

## Celula 5 — Inicializacao da Q-table

A Q-table e uma matriz com uma linha para cada estado e uma coluna para cada acao.

Cada celula `Q[estado][acao]` armazena a estimativa de quao boa e aquela acao naquele estado.

No inicio todos os valores sao zero porque o robo ainda nao tentou nada.

A cada passo do treinamento a formula do Q-Learning atualiza esses valores:

$$Q(s,a) = Q(s,a) + \alpha \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s,a) \right]$$

In [ ]:
# Cria a Q-table como uma matriz de zeros
# Shape (16, 4): 16 linhas (estados 0 a 15), 4 colunas (acoes 0 a 3)
q_table = np.zeros((n_estados, n_acoes))

# Inicializa o epsilon com o valor inicial
epsilon = epsilon_inicio

# Imprime a tabela inicial para confirmar que esta toda zerada
print("Q-table inicial — todos os valores sao zero:")
print()
print(f"{'Estado':>7}  {'Esquerda':>10}  {'Baixo':>10}  {'Direita':>10}  {'Cima':>10}")
print("-" * 58)
for s in range(n_estados):
    print(
        f"  {s:>5d}  "
        f"{q_table[s][0]:>10.4f}  "
        f"{q_table[s][1]:>10.4f}  "
        f"{q_table[s][2]:>10.4f}  "
        f"{q_table[s][3]:>10.4f}"
    )
print()
print("Todos os valores sao 0: o robo nao tem preferencia por nenhuma acao ainda.")

## Celula 6 — Q-table Inicial como Mapa de Calor

Visualizamos a Q-table como um mapa de calor.

Antes do treinamento o mapa e todo uniforme (azul claro), indicando que todos os valores sao iguais (zero).

Apos o treinamento veremos esse mapa bem diferente: as acoes boas serao escuras e as ruins ficarao claras.

In [ ]:
def visualizar_qtable(q_table, titulo="Q-table"):
    """
    Exibe a Q-table como mapa de calor.

    Celulas mais escuras (valores maiores) indicam acoes mais valiosas.
    Cada linha e um estado (0 a 15).
    Cada coluna e uma acao (Esquerda, Baixo, Direita, Cima).
    """

    fig, ax = plt.subplots(figsize=(8, 9))

    # Cria o mapa de calor usando a escala de azul
    # vmin=0 fixa o minimo em zero para comparacao entre graficos
    im = ax.imshow(q_table, cmap="Blues", aspect="auto", vmin=0)

    # Barra de cores lateral explicando a escala
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Valor Q (maior = acao melhor)", fontsize=10)

    # Nomes das acoes no eixo horizontal
    ax.set_xticks(range(n_acoes))
    ax.set_xticklabels(nomes_acoes, fontsize=10)

    # Numeros dos estados no eixo vertical
    ax.set_yticks(range(n_estados))
    ax.set_yticklabels([f"Estado {i}" for i in range(n_estados)], fontsize=9)

    # Valor numerico dentro de cada celula
    for s in range(n_estados):
        for a in range(n_acoes):
            v = q_table[s][a]
            cor = "white" if v > 0.4 else "black"
            ax.text(
                a, s, f"{v:.3f}",
                ha="center", va="center",
                fontsize=8, color=cor
            )

    ax.set_title(titulo, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Acao", fontsize=11)
    ax.set_ylabel("Estado", fontsize=11)

    plt.tight_layout()
    plt.show()


# Exibe a Q-table antes do treinamento
visualizar_qtable(q_table, titulo="Q-table ANTES do treinamento — todos os valores sao zero")

## Celula 7 — Treinamento Principal com Decaimento de Epsilon

Aqui acontece o aprendizado do robo.

Em cada episodio o robo tenta atravessar o tabuleiro desde o inicio.

Em cada passo dentro de um episodio:
1. O robo observa em qual estado esta
2. Decide uma acao usando a estrategia epsilon-greedy
3. Executa a acao e observa o resultado
4. Atualiza a Q-table com a formula do Q-Learning
5. Avanca para o proximo estado

O epsilon vai diminuindo a cada episodio, fazendo o robo explorar cada vez menos e aproveitar mais o que aprendeu.

In [ ]:
# Reinicializa a Q-table do zero para garantir um treinamento limpo
q_table = np.zeros((n_estados, n_acoes))

# Reinicializa o epsilon com o valor inicial
epsilon = epsilon_inicio

# Lista para guardar a recompensa total de cada episodio
# Usaremos para plotar a curva de aprendizado
recompensas_por_episodio = []

# Lista para guardar o valor do epsilon a cada episodio
# Usaremos para plotar o grafico de decaimento
epsilons_por_episodio = []

# Dicionario para guardar caminhos de episodios selecionados
# Chave = numero do episodio, Valor = (caminho, chegou_ao_objetivo)
caminhos_para_exibir = {}

# Loop principal de treinamento
for episodio in range(1, episodios + 1):

    # Reinicia o ambiente: o robo volta para o estado 0
    # env.reset() devolve (estado_inicial, informacoes_extras)
    # O _ descarta as informacoes extras que nao usamos
    estado, _ = env.reset()

    # Indica se este episodio terminou (objetivo ou buraco)
    encerrado = False

    # Acumula a recompensa total recebida neste episodio
    recompensa_total = 0

    # Registra os estados visitados para visualizacao
    caminho = [estado]

    # Loop interno: o robo executa passos ate o episodio terminar
    # O limite de 200 passos evita que o robo fique andando para sempre
    for _ in range(200):

        if encerrado:
            break

        # -----------------------------------------------
        # ESTRATEGIA EPSILON-GREEDY: escolha da acao
        # -----------------------------------------------
        # Sorteamos um numero entre 0.0 e 1.0
        # Se menor que epsilon: acao aleatoria (explorar)
        # Se maior ou igual: melhor acao da Q-table (explotar)
        if random.random() < epsilon:
            # Exploracao: escolhe acao aleatoria para descobrir o ambiente
            # env.action_space.sample() sorteia 0, 1, 2 ou 3
            acao = env.action_space.sample()
        else:
            # Explotacao: escolhe a acao com maior valor na linha do estado
            # np.argmax retorna o indice do maior valor no array
            acao = np.argmax(q_table[estado])

        # -----------------------------------------------
        # EXECUCAO DA ACAO
        # -----------------------------------------------
        # env.step(acao) executa a acao e retorna:
        #   prox_estado : novo estado apos a acao
        #   recompensa  : +1 se chegou ao objetivo, 0 nos outros casos
        #   encerrado   : True se o episodio terminou
        #   truncado    : True se atingiu limite de passos (ignoramos)
        #   _           : informacoes extras (nao usamos)
        prox_estado, recompensa, encerrado, truncado, _ = env.step(acao)

        # -----------------------------------------------
        # ATUALIZACAO DA Q-TABLE: formula do Q-Learning
        # -----------------------------------------------

        # Melhor valor possivel no proximo estado (olha o futuro)
        melhor_futuro = np.max(q_table[prox_estado])

        # Valor atual registrado para este par (estado, acao)
        valor_atual = q_table[estado][acao]

        # Aplica a formula do Q-Learning:
        # novo = antigo + alpha * (recompensa + gamma * melhor_futuro - antigo)
        # O termo entre parenteses e o erro de predicao
        # Se positivo: acao foi melhor que o esperado, valor sobe
        # Se negativo: acao foi pior que o esperado, valor desce
        novo_valor = valor_atual + alpha * (
            recompensa + gamma * melhor_futuro - valor_atual
        )

        # Atualiza a celula correspondente da Q-table
        q_table[estado][acao] = novo_valor

        # Avanca para o proximo estado
        estado = prox_estado
        recompensa_total += recompensa
        caminho.append(estado)

    # -----------------------------------------------
    # DECAIMENTO DO EPSILON
    # -----------------------------------------------
    # A cada episodio, reduzimos epsilon multiplicando por (1 - taxa_decaimento)
    # max(..., epsilon_fim) garante que epsilon nunca fique abaixo do minimo
    epsilon = max(epsilon_fim, epsilon * (1 - taxa_decaimento))

    # Guarda os valores para os graficos
    recompensas_por_episodio.append(recompensa_total)
    epsilons_por_episodio.append(epsilon)

    # Salva o caminho nos episodios selecionados para visualizacao
    if episodio == 1 or episodio % intervalo_visual == 0:
        caminhos_para_exibir[episodio] = (caminho, recompensa_total > 0)

# Calcula a taxa de sucesso nos ultimos 100 episodios
sucesso_final = np.mean(recompensas_por_episodio[-100:])

print(f"Treinamento concluido: {episodios} episodios.")
print(f"Epsilon inicial: {epsilon_inicio:.2f} => Epsilon final: {epsilon:.4f}")
print(f"Taxa de sucesso nos ultimos 100 episodios: {sucesso_final * 100:.1f}%")

## Celula 8 — Grafico do Decaimento do Epsilon

Este grafico mostra como o epsilon diminuiu ao longo do treinamento.

No inicio o robo explorava quase sempre (epsilon proximo de 1.0).

Com o tempo foi reduzindo a exploracao e passando a confiar mais na Q-table.

No final o robo quase sempre usa o que aprendeu (epsilon proximo de 0.01).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Plota a curva de decaimento do epsilon
ax.plot(
    range(1, episodios + 1),
    epsilons_por_episodio,
    color="#8e44ad",  # roxo
    linewidth=2
)

# Linha tracejada mostrando o epsilon minimo
ax.axhline(
    y=epsilon_fim, color="#e74c3c",
    linestyle="--", linewidth=1.5,
    label=f"Epsilon minimo = {epsilon_fim}"
)

# Anotacao explicando a fase de exploracao
ax.annotate(
    "Exploracao alta:\nrobo testa acoes aleatorias",
    xy=(200, 0.85), fontsize=9, color="#2c3e50",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f8f9fa", edgecolor="#bdc3c7")
)

# Anotacao explicando a fase de explotacao
ax.annotate(
    "Explotacao alta:\nrobo usa o que aprendeu",
    xy=(episodios - 1500, 0.06), fontsize=9, color="#2c3e50",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#f8f9fa", edgecolor="#bdc3c7")
)

ax.set_xlabel("Episodio", fontsize=12)
ax.set_ylabel("Epsilon", fontsize=12)
ax.set_title(
    "Decaimento do Epsilon ao longo do treinamento",
    fontsize=13, fontweight="bold"
)
ax.set_ylim(-0.05, 1.1)
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Epsilon no episodio 1:           {epsilons_por_episodio[0]:.4f}")
print(f"Epsilon no episodio {episodios // 2:>5}:       {epsilons_por_episodio[episodios // 2 - 1]:.4f}")
print(f"Epsilon no episodio {episodios:>5} (final): {epsilons_por_episodio[-1]:.4f}")

## Celula 9 — Evolucao do Caminho ao Longo do Treinamento

Aqui vemos como o comportamento do robo foi mudando ao longo do treinamento.

No episodio 1, o robo age aleatoriamente e provavelmente cai num buraco logo no inicio.

Nos episodios intermediarios, o robo comeca a encontrar caminhos melhores.

No ultimo episodio exibido, o robo ja deve estar seguindo um caminho seguro e direto.

In [ ]:
# Exibe os mapas com os caminhos salvos durante o treinamento
# Cada mapa mostra o trajeto percorrido num episodio especifico
for num_ep, (caminho, chegou) in sorted(caminhos_para_exibir.items()):
    resultado = "chegou ao objetivo" if chegou else "nao chegou ao objetivo"
    titulo = f"Episodio {num_ep} — {resultado}"
    desenhar_mapa(mapa, titulo=titulo, caminho=caminho)

## Celula 10 — Q-table Apos o Treinamento

Agora a Q-table esta preenchida com valores que refletem o que o robo aprendeu.

Compare este mapa de calor com o que vimos antes do treinamento (todo uniforme).

As celulas mais escuras sao as acoes mais valiosas em cada estado.

Para cada estado, a coluna mais escura indica a melhor acao a tomar.

In [ ]:
# Exibe a Q-table apos o treinamento
visualizar_qtable(
    q_table,
    titulo="Q-table APOS o treinamento — valores aprendidos"
)

# Imprime tambem em formato texto para leitura detalhada
print("Q-table final:")
print()
print(f"{'Estado':>7}  {'Esquerda':>10}  {'Baixo':>10}  {'Direita':>10}  {'Cima':>10}  {'Melhor acao':>14}")
print("-" * 75)
for s in range(n_estados):
    melhor = np.argmax(q_table[s])
    print(
        f"  {s:>5d}  "
        f"{q_table[s][0]:>10.4f}  "
        f"{q_table[s][1]:>10.4f}  "
        f"{q_table[s][2]:>10.4f}  "
        f"{q_table[s][3]:>10.4f}  "
        f"{nomes_acoes[melhor]:>14}"
    )

## Celula 11 — Politica Otima com Setas no Mapa

A politica otima e o resultado final do treinamento.

Para cada estado escolhemos a acao com o maior valor na Q-table:

$$\pi(s) = \arg\max_a \, Q(s, a)$$

Isso nos da uma seta em cada celula segura, indicando exatamente o que o robo deve fazer.

Seguindo as setas do inicio ao fim, o robo chega ao objetivo sem errar.

In [ ]:
def visualizar_politica(q_table, mapa, titulo="Politica aprendida"):
    """
    Desenha o tabuleiro com uma seta em cada celula segura.

    A seta indica a melhor acao segundo a Q-table treinada.
    A melhor acao e: pi(s) = argmax_a Q(s, a).
    Buracos e objetivo nao tem seta.
    """

    fig, ax = plt.subplots(figsize=(7, 8))

    cores_celula = {
        "S": "#2ecc71",
        "F": "#ecf0f1",
        "H": "#e74c3c",
        "G": "#3498db",
    }

    for linha_idx, linha in enumerate(mapa):
        for col_idx, celula in enumerate(linha):

            estado = linha_idx * 4 + col_idx

            ret = plt.Rectangle(
                (col_idx, 3 - linha_idx), 1, 1,
                color=cores_celula[celula], ec="#7f8c8d", lw=2
            )
            ax.add_patch(ret)

            # Numero do estado no canto
            ax.text(
                col_idx + 0.07, 3 - linha_idx + 0.87,
                str(estado), fontsize=9, color="#555555", va="top"
            )

            if celula in ("H", "G"):
                # Buracos e objetivo: mostra apenas a letra
                ax.text(
                    col_idx + 0.5, 3 - linha_idx + 0.48,
                    celula, fontsize=28, ha="center", va="center",
                    fontweight="bold",
                    color="white" if celula == "H" else "white"
                )
            else:
                # Celulas seguras: seta da melhor acao e nome abaixo
                # np.argmax retorna o indice do maior valor na linha do estado
                melhor_acao = np.argmax(q_table[estado])

                # Simbolo da acao: <, v, >, ^
                ax.text(
                    col_idx + 0.5, 3 - linha_idx + 0.52,
                    simbolos_acoes[melhor_acao],
                    fontsize=30, ha="center", va="center",
                    fontweight="bold", color="#2c3e50"
                )

                # Nome da acao em texto menor
                ax.text(
                    col_idx + 0.5, 3 - linha_idx + 0.10,
                    nomes_acoes[melhor_acao],
                    fontsize=7, ha="center", va="bottom",
                    color="#555555"
                )

    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titulo, fontsize=13, fontweight="bold", pad=12)

    legenda = [
        mpatches.Patch(color="#2ecc71", label="S — Inicio"),
        mpatches.Patch(color="#ecf0f1", label="F — Gelo seguro (seta = melhor acao)"),
        mpatches.Patch(color="#e74c3c", label="H — Buraco"),
        mpatches.Patch(color="#3498db", label="G — Objetivo"),
    ]
    ax.legend(
        handles=legenda, loc="upper center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=1, fontsize=10, frameon=True, edgecolor="#bdc3c7"
    )

    plt.tight_layout()
    plt.show()


# Exibe o mapa com a politica otima aprendida
visualizar_politica(
    q_table, mapa,
    titulo="Politica otima aprendida pelo robo"
)

# Imprime a politica por estado em formato texto
print("Politica otima por estado:")
print()
tipos = {"S": "Inicio   ", "F": "Gelo     ", "H": "Buraco   ", "G": "Objetivo "}
for li, linha in enumerate(mapa):
    for ci, c in enumerate(linha):
        s = li * 4 + ci
        if c in ("H", "G"):
            print(f"  Estado {s:>2d} ({tipos[c]}): {c}")
        else:
            m = np.argmax(q_table[s])
            print(f"  Estado {s:>2d} ({tipos[c]}): {nomes_acoes[m]} {simbolos_acoes[m]}")

## Celula 12 — Curva de Aprendizado com as Tres Fases

Este grafico mostra como o desempenho do robo evoluiu ao longo dos episodios.

Observe as tres fases:
- **Fase 1 — Exploracao**: o robo ainda nao encontrou caminhos bons, a curva e baixa
- **Fase 2 — Aprendizado**: o robo comeca a encontrar o objetivo com mais frequencia
- **Fase 3 — Convergencia**: a curva se estabiliza, o robo aprendeu a politica otima

In [ ]:
# Tamanho do bloco para calcular a media movel
# Media de 100 episodios consecutivos suaviza as oscilacoes
bloco = 100

# Calcula a media de recompensa a cada bloco de episodios
medias = [
    np.mean(recompensas_por_episodio[i: i + bloco])
    for i in range(0, len(recompensas_por_episodio), bloco)
]

# Eixo x: numero do episodio central de cada bloco
eixo_x = [i * bloco + bloco // 2 for i in range(len(medias))]

fig, ax = plt.subplots(figsize=(11, 5))

# Curva de aprendizado
ax.plot(
    eixo_x, medias,
    color="#2980b9", linewidth=2.5,
    marker="o", markersize=4,
    label=f"Recompensa media (blocos de {bloco})"
)

# Linha do maximo teorico
ax.axhline(
    y=1.0, color="#27ae60",
    linestyle="--", linewidth=1.5,
    label="Maximo teorico (taxa de sucesso = 100%)"
)

# Anotacao da Fase 1
ax.annotate(
    "Fase 1\nExploracao\n(epsilon alto)",
    xy=(eixo_x[2], medias[2]),
    xytext=(eixo_x[2] + 200, medias[2] + 0.15),
    fontsize=9, color="#c0392b",
    arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2)
)

# Anotacao da Fase 2 — posicionada no trecho de subida rapida
idx_meio = len(medias) // 3
ax.annotate(
    "Fase 2\nAprendizado rapido",
    xy=(eixo_x[idx_meio], medias[idx_meio]),
    xytext=(eixo_x[idx_meio] + 300, medias[idx_meio] - 0.20),
    fontsize=9, color="#e67e22",
    arrowprops=dict(arrowstyle="->", color="#e67e22", lw=1.2)
)

# Anotacao da Fase 3 — convergencia no final
ax.annotate(
    "Fase 3\nConvergencia\n(politica otima)",
    xy=(eixo_x[-4], medias[-4]),
    xytext=(eixo_x[-4] - 1000, medias[-4] - 0.25),
    fontsize=9, color="#27ae60",
    arrowprops=dict(arrowstyle="->", color="#27ae60", lw=1.2)
)

ax.set_xlabel("Episodio", fontsize=12)
ax.set_ylabel("Taxa de sucesso", fontsize=12)
ax.set_title(
    "Curva de aprendizado — como o robo foi melhorando",
    fontsize=13, fontweight="bold"
)
ax.set_ylim(-0.05, 1.15)
ax.set_xlim(0, episodios)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=10, loc="lower right")

plt.tight_layout()
plt.show()

print(f"Taxa de sucesso final (ultimos 100 episodios): {sucesso_final * 100:.1f}%")

## Celula 13 — Simulacao Passo a Passo com o Robo Treinado

Agora simulamos o robo usando apenas a politica otima que aprendeu.

Nesta simulacao nao ha mais exploracao (epsilon = 0): o robo sempre escolhe a melhor acao da Q-table.

Para cada passo imprimimos o estado atual, a acao escolhida, o resultado e o proximo estado.

Ao final exibimos o mapa com o caminho percorrido.

In [ ]:
# Reinicia o ambiente para uma simulacao limpa
estado, _ = env.reset()

encerrado = False
passo = 0
caminho_final = [estado]
recompensa_total = 0

print("Simulacao do robo usando a politica otima (sem exploracao)")
print()
print(f"{'Passo':>6}  {'Estado':>7}  {'Acao':>14}  {'Resultado':>10}  {'Proximo':>8}")
print("-" * 58)

# Limite de 50 passos para evitar loop infinito em caso de politica imperfeita
for _ in range(50):

    if encerrado:
        break

    passo += 1

    # Escolhe sempre a melhor acao da Q-table (sem exploracao)
    acao = np.argmax(q_table[estado])

    # Executa a acao
    prox, recompensa, encerrado, truncado, _ = env.step(acao)

    recompensa_total += recompensa
    caminho_final.append(prox)

    # Descricao do resultado deste passo
    if recompensa > 0:
        resultado_str = "chegou (+1)"
    elif encerrado:
        resultado_str = "buraco  (0)"
    else:
        resultado_str = "seguiu  (0)"

    print(
        f"  {passo:>4d}  "
        f"  Estado {estado:>2d}  "
        f"  {nomes_acoes[acao]:>12}  "
        f"  {resultado_str:>10}  "
        f"  Estado {prox:>2d}"
    )

    estado = prox

print("-" * 58)
print()
if recompensa_total > 0:
    print(f"Robo chegou ao objetivo em {passo} passos.")
else:
    print(f"Robo nao chegou ao objetivo em {passo} passos.")
print(f"Sequencia de estados: {caminho_final}")

## Celula 14 — Mapa do Caminho Final

Visualizamos o caminho exato que o robo percorreu na simulacao acima.

As celulas laranjas mostram por onde o robo passou.

Compare este caminho com o mapa da politica otima da celula anterior.

In [ ]:
# Exibe o mapa com o caminho percorrido pelo robo treinado
desenhar_mapa(
    mapa,
    titulo="Caminho percorrido pelo robo treinado",
    caminho=caminho_final
)

## Celula 15 — Experimento: Comparacao de Hiperparametros

Esta celula treina varios agentes com configuracoes diferentes e compara as curvas de aprendizado.

Observe como cada hiperparametro afeta a velocidade e a qualidade do aprendizado:

- **Alpha baixo**: correcoes menores a cada passo, curva sobe mais devagar
- **Gamma baixo**: robo nao valoriza o futuro, pode nao aprender a chegar ao objetivo
- **Sem decaimento**: epsilon fixo alto, muita exploracao, converge mais lentamente
- **Sem exploracao**: epsilon sempre zero desde o inicio, robo nunca descobre novos caminhos

In [ ]:
def treinar_agente(alpha, gamma, ep_inicio, ep_fim, decaimento, episodios):
    """
    Treina um novo agente com os hiperparametros fornecidos.

    Retorna a lista de recompensas por episodio.
    Cria ambiente e Q-table novos a cada chamada para comparacao justa.
    """

    amb = gym.make("FrozenLake-v1", is_slippery=False)
    tab = np.zeros((amb.observation_space.n, amb.action_space.n))
    eps = ep_inicio
    recompensas = []

    for _ in range(episodios):
        s, _ = amb.reset()
        encerrado = False
        rt = 0

        for _ in range(200):
            if encerrado:
                break

            # Estrategia epsilon-greedy
            if random.random() < eps:
                a = amb.action_space.sample()
            else:
                a = np.argmax(tab[s])

            ns, r, encerrado, _, _ = amb.step(a)

            # Atualizacao Q-Learning
            tab[s, a] += alpha * (r + gamma * np.max(tab[ns]) - tab[s, a])

            s = ns
            rt += r

        # Decaimento do epsilon ao final de cada episodio
        eps = max(ep_fim, eps * (1 - decaimento))
        recompensas.append(rt)

    return recompensas


# Configuracoes dos experimentos a comparar
# Cada entrada descreve um conjunto de hiperparametros
experimentos = [
    {
        "nome": "Padrao (a=0.8, g=0.99, decaimento)",
        "alpha": 0.8, "gamma": 0.99,
        "ep_inicio": 1.0, "ep_fim": 0.01, "decaimento": 0.001,
        "cor": "#2980b9"
    },
    {
        "nome": "Alpha baixo (a=0.1)",
        "alpha": 0.1, "gamma": 0.99,
        "ep_inicio": 1.0, "ep_fim": 0.01, "decaimento": 0.001,
        "cor": "#27ae60"
    },
    {
        "nome": "Gamma baixo (g=0.5)",
        "alpha": 0.8, "gamma": 0.50,
        "ep_inicio": 1.0, "ep_fim": 0.01, "decaimento": 0.001,
        "cor": "#e74c3c"
    },
    {
        "nome": "Sem decaimento (epsilon fixo 0.1)",
        "alpha": 0.8, "gamma": 0.99,
        "ep_inicio": 0.1, "ep_fim": 0.1, "decaimento": 0.0,
        "cor": "#f39c12"
    },
    {
        "nome": "Sem exploracao (epsilon = 0)",
        "alpha": 0.8, "gamma": 0.99,
        "ep_inicio": 0.0, "ep_fim": 0.0, "decaimento": 0.0,
        "cor": "#8e44ad"
    },
]

ep_comp = 5000
bloco_comp = 100

fig, ax = plt.subplots(figsize=(12, 6))

for exp in experimentos:

    # Treina um agente com as configuracoes deste experimento
    recs = treinar_agente(
        alpha=exp["alpha"],
        gamma=exp["gamma"],
        ep_inicio=exp["ep_inicio"],
        ep_fim=exp["ep_fim"],
        decaimento=exp["decaimento"],
        episodios=ep_comp
    )

    # Media movel para suavizar a curva
    medias_c = [
        np.mean(recs[i: i + bloco_comp])
        for i in range(0, len(recs), bloco_comp)
    ]
    eixo_c = [i * bloco_comp + bloco_comp // 2 for i in range(len(medias_c))]

    ax.plot(
        eixo_c, medias_c,
        color=exp["cor"], linewidth=2,
        label=exp["nome"]
    )

# Linha de referencia do maximo
ax.axhline(y=1.0, color="black", linestyle=":", linewidth=1, label="Maximo teorico")

ax.set_xlabel("Episodio", fontsize=12)
ax.set_ylabel("Taxa de sucesso", fontsize=12)
ax.set_title(
    "Comparacao de hiperparametros — efeito no aprendizado",
    fontsize=13, fontweight="bold"
)
ax.set_ylim(-0.05, 1.15)
ax.set_xlim(0, ep_comp)
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.show()

print("O que observar:")
print("  Alpha baixo     : curva sobe mais devagar (correcoes menores a cada passo)")
print("  Gamma baixo     : robo nao ve longe, pode nao aprender a chegar ao objetivo")
print("  Sem decaimento  : epsilon fixo causa mais oscilacao na convergencia")
print("  Sem exploracao  : robo nunca descobre caminhos novos, nao aprende")

## Celula 16 — Resumo Final

Esta celula imprime um resumo estruturado de tudo que aconteceu neste notebook.

Use como referencia para revisar os conceitos da aula.

In [ ]:
print("================================================")
print(" RESUMO DA AULA — Q-Learning no FrozenLake")
print("================================================")
print()
print("AMBIENTE:")
print(f"  Tabuleiro: 4x4 = {n_estados} estados numerados de 0 a 15")
print(f"  Acoes possiveis: {n_acoes} (esquerda, baixo, direita, cima)")
print("  Buracos nos estados 5, 7, 11 e 12")
print("  Objetivo no estado 15")
print()
print("HIPERPARAMETROS UTILIZADOS:")
print(f"  alpha (taxa de aprendizado)  = {alpha}")
print(f"  gamma (fator de desconto)    = {gamma}")
print(f"  epsilon inicio               = {epsilon_inicio}")
print(f"  epsilon fim                  = {epsilon_fim}")
print(f"  taxa de decaimento           = {taxa_decaimento}")
print(f"  episodios                    = {episodios}")
print()
print("RESULTADO:")
print(f"  Taxa de sucesso final: {sucesso_final * 100:.1f}%")
print()
print("POLITICA OTIMA (mapa de setas):")
for li, linha in enumerate(mapa):
    row = "  "
    for ci, c in enumerate(linha):
        s = li * 4 + ci
        row += f" {c if c in 'HG' else simbolos_acoes[np.argmax(q_table[s])]} "
    print(row)
print()
print("CONCEITOS REVISADOS NESTE NOTEBOOK:")
print("  Agente         = o robo que toma decisoes")
print("  Ambiente       = o tabuleiro FrozenLake")
print("  Estado         = numero da celula onde o robo esta")
print("  Acao           = direcao escolhida (esquerda, baixo, direita, cima)")
print("  Recompensa     = +1 ao chegar no objetivo, 0 nos outros casos")
print("  Q-table        = tabela com o valor estimado de cada acao em cada estado")
print("  Q-Learning     = formula que atualiza a Q-table a cada passo")
print("  Epsilon-greedy = estrategia que equilibra exploracao e explotacao")
print("  Decaimento     = reducao gradual do epsilon ao longo do treinamento")
print("  Politica otima = melhor acao em cada estado apos o treinamento")
print("  Convergencia   = ponto em que o robo para de melhorar")
print()
print("================================================")